## Notebook 03 — SageMaker Pipeline with Post-Pipeline SageMaker MLflow App Logging

Module: ITI113 Machine Learning & Operations
Focus Area: C — MLOps (Pipeline, experiment tracking, CI/CD, deployment)
Estimated Runtime: 15–25 minutes for the pipeline execution

## What this notebook does
1. Writes preprocess.py, train.py, and inference.py to a local src/ folder.
2. Defines and runs a SageMaker Pipeline: Process → Train → Condition → Register.
3. Keeps the SageMaker training container free of MLflow credentials and MLflow dependencies.
4. After a successful pipeline run, the notebook reads SageMaker job metadata and metrics, then logs them to the team’s SageMaker Serverless MLflow App. The notebook validates the app TeamId tag before logging.
5. Registers a quality-approved model in SageMaker Model Registry.
This separation makes troubleshooting safer and simpler:

SageMaker Pipeline                    Notebook-side MLflow logging
Process → Train → Gate → Register     Read run metadata → Log to MLflow App
This version follows Notebook 01/02 and uses the SageMaker MLflow App ARN when available from mlflow_app_config_team01_s004.json.

In [1]:
# %pip uninstall -y sagemaker sagemaker-core sagemaker-mlops sagemaker-serve sagemaker-train

In [2]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

  Using cached sagemaker-2.257.6-py3-none-any.whl.metadata (19 kB)


  Using cached boto3-1.43.80-py3-none-any.whl.metadata (6.6 kB)


  Using cached botocore-1.43.80-py3-none-any.whl.metadata (5.6 kB)


  Using cached mlflow-3.15.2-py3-none-any.whl.metadata (49 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 12/13 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 12/13 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 12/13 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [mlflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 3.8.0 requires botocore<1.43.47,>=1.43.3, but you have botocore 1.43.80 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
sagemaker-mlops 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is inco

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

In [3]:
import boto3
import sagemaker
import json
import os
import time
from pathlib import Path

# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

# Use the ITI113 course bucket and team prefix.
# This matches the team execution role S3 policy, e.g.:
# s3://nyp-26s1-iti113/iti113/team40/
BUCKET  = "nyp-26s1-iti113"

# Change these for the current student/profile.
TEAM_ID = "team14"
STUDENT_ID = "s1401"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "airbnb-instant-booking"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# Instance types.
# If ml.m5.large quota is 0, change these to an approved available training/processing type.
# For ITI113, keep Studio spaces on ml.t3.medium and use SageMaker jobs for training/processing.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ----------------------------
# SageMaker Serverless MLflow App setup
# ----------------------------
# Preferred:
# 1. Team-level config copied/shared from Notebook 01A:
#       mlflow_app_config_team40.json
# 2. Student-specific config from Notebook 01A:
#       mlflow_app_config_team40_s4002.json
# 3. Any local config matching this team:
#       mlflow_app_config_team40_*.json
#
# Important:
# Do not use another team's MLflow ARN. With team-level IAM
# restriction, wrong-team access should fail with 403.
# ----------------------------

TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/InstantBooking-Classification"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        config_used = config_file
        break

# Fallback for classroom testing only.
# Update this to your team's MLflow App ARN from Notebook 01A if the config file
# is not available in this Studio workspace.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-PIGMOQJH46PS"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

# Validate config team, if present.
config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

# ----------------------------
# Safety check for team-level MLflow restriction
# ----------------------------
# The selected MLflow App must have ResourceTag/TeamId = TEAM_ID.
# If a student accidentally uses another team's ARN, this should either:
# - fail with AccessDenied / 403 due to IAM restriction, or
# - fail this explicit validation before logging.
# ----------------------------

sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

# Optional: store for downstream cells and subprocesses.
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME


# ----------------------------
# MLflow App UI link helpers
# ----------------------------
# MLflow may print generic links such as:
# https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...
# Those links are not presigned and may show a permission/session error.
# Use these helpers to generate a fresh presigned SageMaker MLflow App URL
# and append the experiment/run fragment.

def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    # Remove any existing fragment before appending our own MLflow UI route.
    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME       = f"iti113-{TEAM_ID}-airbnb-instant-booking"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-airbnb-instant-booking"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-airbnb-instant-booking"
QUALITY_GATE_AUC    = 0.75

RAW_DATA_URI = f"s3://{BUCKET}/{PREFIX}/raw/Listings.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

# Store the pipeline source files in S3 first, then download them into a clean local folder.
SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"MLflow App ARN          : {MLFLOW_APP_ARN}")
print(f"MLflow experiment       : {MLFLOW_EXPERIMENT_NAME}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


[WARNING] No local MLflow config file found. Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team.


MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-bigthb6rme9d
  ProjectName: airbnb-listings
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-bigthb6rme9d/team14-shared-powered
  Course: ITI113
  TeamId: team14
  CreatedByNotebook: 01A_setup_sagemaker_mlflow_app
  StudentId: s1402
[OK] MLflow App tag TeamId=team14 matches notebook TEAM_ID=team14
Pipeline                : iti113-team14-airbnb-instant-booking
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team14/data/airbnb-instant-booking
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team14
MLflow App ARN          : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS
MLflow experiment       : ITI113/team14/InstantBooking-Classification
Pipeline source S3 URI  : s3://nyp-26s1-iti113/iti1

## 0A. Precheck SageMaker MLflow App connection

Before launching the SageMaker Pipeline, test that this notebook can create/use the current team's MLflow experiment in the SageMaker MLflow App.

Example for Team 14:

ITI113/team14/Experiment0
This notebook now checks that the selected MLflow App has the correct TeamId tag before logging. If this fails, resolve the MLflow App ARN, sagemaker-mlflow package, or IAM permissions before continuing to the SageMaker Pipeline.

## Note about MLflow links
The MLflow client may print links such as https://mlflow.sagemaker.ap-southeast-1.app.aws/#/.... Those generic links are not presigned and may show a SageMaker MLflow permission/session error. This notebook generates fresh presigned MLflow App URLs using create_presigned_mlflow_app_url() after each logging step. Use those printed presigned URLs instead.

In [4]:
import mlflow
import time

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "Listing",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=precheck_experiment_id,
    run_id=precheck_run_id
)

🏃 View run team14_pipeline_notebook_precheck_1787744997 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2/runs/4d80c4639af84e48ac396ac3a5f6c2ce
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS
Experiment: ITI113/team14/InstantBooking-Classification
Experiment ID: 2
Run ID: 4d80c4639af84e48ac396ac3a5f6c2ce

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjJESkdQUCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNEE0bUtaci91eVJCYXRhbS9XN3I2eEpRanZSTTBPUWw2TjVKWjQvcGw2UFlBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGbmVqYzFWVWd6Y25sUGJWUlpXVXg0ZUV0bmRIa3lSRXBNZFhSUFdXdHhkbm8xZW1sdlJXUnphbU5DWlZsT1kwd3JTRXg0VkdocGFrNHhVa2d3YTFSa1VUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFiWVpPYVp6eGlWQTlueE5OWHh6N2U4QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4WmtYOHZSbW1LaVJSMDRYa0NBUkNBT3lHaDJ1MEl5ZTUvS0pydjEwUDh5TUtsMi93V284ejhMc1BqUTlscExjeG5YOWtNTld5UXdjR2h0NU5NNEdnSEljNWVjSnBNYXhQODFUV1FBZ0FBRUFCaGlER3pTanBUNDBRcHBjNXBuL1MwM0s4Vm44NlIwWmk5RTFWdC9aZUJDTGE4NmdsWjZjRV


Presigned MLflow run URL:
https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjVPUlcySCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHc5SDFXa3g3dGs1N3J6K2NTNmM5Tm1Hak8ya2Z5ZHlqanVKNVlZeFZCTzRBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFcmJHaHVZbFJ1VTFkdlZEaFZNa3BaZVdkQlkxZDJWVEoyT0VobFMweDZWbEZhWVdoVFltcHRNekZNWW05aU4yOUNSRFJVZUVkRlNFZGtXVlpYUldOdlFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFUMFcvZDNTWmxFYlhjbHdnWWFPWjRnQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6TGljRm1mU1Z0M1U2Mmduc0NBUkNBTzdpTG9HeHVxRWNiOUxncVlpVHZWWlZnUTRjRmpiV2t3N3NHa01xRnE5YVBoNVBGV2xhOHFJL21GSHFJWno1aUM3d1FqVjZSQVF4TTVhTTVBZ0FBRUFCb2VlMWFjTzNiQkZuU01uUTR5V3FSeTljc0d0TXR6RW9jTTZYT09VQlNYOVhLSEZtYWFiUGVLYVFL

## 1. Write Pipeline Scripts

SageMaker Pipeline steps run as isolated jobs, each in its own managed container.

The training script deliberately does not import or call MLflow. It only trains the model, prints quality metrics for the SageMaker quality gate, and saves model.pkl for SageMaker Model Registry.

After the pipeline completes, this notebook logs the completed run to the team SageMaker MLflow App experiment from the notebook environment. This means the MLflow App access stays in the notebook session and is not sent to the SageMaker training container.

After writing these files locally, the next sections upload them to S3 and download them into pipeline_src/. The pipeline then uses pipeline_src/, not the original src/ folder.

In [5]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')

src/ directory ready


In [6]:
%%writefile src/preprocess.py
# ==============================================================================
# preprocess.py
# Airbnb Instant Booking - Binary Classification
#
# STANDALONE SAGEMAKER PROCESSING SCRIPT
#
# PURPOSE
# -------
# 1. Load raw Airbnb CSV directly from SageMaker Processing input.
# 2. Perform governance-driven cleaning and feature engineering.
# 3. Perform stratified train/test split.
# 4. Fit preprocessing ONLY on training data.
# 5. Save train.csv, test.csv, feature_schema.json and
#    preprocessing_artifacts.json.
#
# IMPORTANT
# ---------
# One-Hot Encoding and Scaling have been removed from this step and are now 
# handled natively inside the Scikit-Learn Pipeline in train.py and inference.py.
# ==============================================================================

import os
import re
import json
import glob
import sys
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


# ==============================================================================
# FLUSHED LOGGING
# ==============================================================================

def log(message=""):
    print(message, flush=True)


# ==============================================================================
# CONFIGURATION
# ==============================================================================

TARGET_COLUMN = "instant_bookable"
TARGET_POSITIVE_CLASS = 1

TEST_SIZE = 0.20
RANDOM_STATE = 42

TOP_K_AMENITIES = 30

PRICE_CAP_QUANTILE = 0.995
COUNT_CAP_QUANTILE = 0.995

COORD_DECIMALS = 2

SNAPSHOT_DATE = pd.Timestamp(
    os.environ.get(
        "SNAPSHOT_DATE",
        "2026-08-16"
    )
)

INPUT_DIR = os.environ.get(
    "SM_INPUT_DIR",
    "/opt/ml/processing/input"
)

OUTPUT_DIR = os.environ.get(
    "SM_OUTPUT_DIR",
    "/opt/ml/processing/output"
)


# ==============================================================================
# STARTUP
# ==============================================================================

log("=" * 80)
log("AIRBNB CLASSIFICATION PREPROCESSING")
log("=" * 80)
log(f"Python version       : {sys.version.split()[0]}")
log(f"Input directory      : {INPUT_DIR}")
log(f"Output directory     : {OUTPUT_DIR}")
log(f"Random state         : {RANDOM_STATE}")
log(f"Test size            : {TEST_SIZE}")
log("=" * 80)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ==============================================================================
# 1. FIND RAW INPUT CSV
# ==============================================================================

log("\n" + "=" * 80)
log("1. LOCATING RAW INPUT DATA")
log("=" * 80)


csv_candidates = sorted(
    glob.glob(
        os.path.join(
            INPUT_DIR,
            "**",
            "*.csv"
        ),
        recursive=True
    )
)


if not csv_candidates:
    raise FileNotFoundError(
        f"No CSV file found under {INPUT_DIR}"
    )


log("CSV files found:")

for path in csv_candidates:
    log(f"  - {path}")


# Prefer a file that looks like the Airbnb raw dataset.
preferred = [
    p for p in csv_candidates
    if "airbnb" in os.path.basename(p).lower()
    or "listing" in os.path.basename(p).lower()
]


if preferred:
    RAW_INPUT_PATH = preferred[0]
else:
    RAW_INPUT_PATH = csv_candidates[0]


log(f"\nSelected input: {RAW_INPUT_PATH}")


# ==============================================================================
# 2. LOAD RAW DATA
# ==============================================================================
log("\n" + "=" * 80)
log("2. LOADING RAW DATA")
log("=" * 80)

log(f"Loading: {RAW_INPUT_PATH}")
log("CSV encoding        : latin1")

df_raw = pd.read_csv(
    RAW_INPUT_PATH,
    encoding="latin1",
    low_memory=False
)

log(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

log("\nRaw columns:")

for i, col in enumerate(df_raw.columns, start=1):
    log(f"{i:03d}. {col}")

# ==============================================================================
# 3. REQUIRED COLUMN CHECK
# ==============================================================================

log("\n" + "=" * 80)
log("3. VALIDATING REQUIRED COLUMNS")
log("=" * 80)


required_columns = [
    "listing_id",
    "host_id",
    "host_since",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_total_listings_count",
    "host_has_profile_pic",
    "host_identity_verified",
    "neighbourhood",
    "city",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "amenities",
    "price",
    "minimum_nights",
    "maximum_nights",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "instant_bookable"
]


missing_required = [
    c for c in required_columns
    if c not in df_raw.columns
]


if missing_required:
    raise ValueError(
        "Required columns are missing:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_required
        )
    )


log("✓ All required columns are present.")


# ==============================================================================
# 4. BASIC GOVERNANCE CLEANING
# ==============================================================================

log("\n" + "=" * 80)
log("4. GOVERNANCE-DRIVEN CLEANING")
log("=" * 80)


df = df_raw.copy()


# ------------------------------------------------------------------------------
# 4.1 Remove exact duplicates
# ------------------------------------------------------------------------------

before = len(df)

df = df.drop_duplicates(
    keep="first"
).reset_index(
    drop=True
)

log(
    f"Exact duplicate rows removed: "
    f"{before - len(df):,}"
)


# ------------------------------------------------------------------------------
# 4.2 Remove duplicate listing IDs
# ------------------------------------------------------------------------------

before = len(df)

df = df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
).reset_index(
    drop=True
)

log(
    f"Duplicate listing_id rows removed: "
    f"{before - len(df):,}"
)


# ------------------------------------------------------------------------------
# 4.3 Target mapping
# ------------------------------------------------------------------------------

df[TARGET_COLUMN] = (
    df[TARGET_COLUMN]
    .astype(str)
    .str.lower()
    .str.strip()
    .map(
        {
            "t": 1,
            "true": 1,
            "1": 1,
            "f": 0,
            "false": 0,
            "0": 0
        }
    )
)


before = len(df)

df = df[
    df[TARGET_COLUMN].notna()
].copy()

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

log(
    f"Rows with invalid/missing target removed: "
    f"{before - len(df):,}"
)


# ------------------------------------------------------------------------------
# 4.4 Numeric conversion
# ------------------------------------------------------------------------------

numeric_columns = [
    "host_response_rate",
    "host_acceptance_rate",
    "host_total_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bedrooms",
    "price",
    "minimum_nights",
    "maximum_nights",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]


for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ------------------------------------------------------------------------------
# 4.5 Convert percentage strings
# ------------------------------------------------------------------------------

for col in [
    "host_response_rate",
    "host_acceptance_rate"
]:

    if col in df.columns:

        # Handles values such as "95%"
        if df[col].dtype == object:

            df[col] = (
                df[col]
                .astype(str)
                .str.replace(
                    "%",
                    "",
                    regex=False
                )
            )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# Convert percentage values from 0-100 to 0-1.
for col in [
    "host_response_rate",
    "host_acceptance_rate"
]:

    if col in df.columns:

        mask = df[col] > 1

        df.loc[
            mask,
            col
        ] = (
            df.loc[
                mask,
                col
            ] / 100.0
        )


# ------------------------------------------------------------------------------
# 4.6 Price validation
# ------------------------------------------------------------------------------

df["price"] = pd.to_numeric(
    df["price"],
    errors="coerce"
)

before = len(df)

df = df[
    df["price"].notna()
    & (df["price"] >= 0)
].copy()

log(
    f"Invalid price rows removed: "
    f"{before - len(df):,}"
)


# ------------------------------------------------------------------------------
# 4.7 Coordinate privacy coarsening
# ------------------------------------------------------------------------------

df["latitude"] = df[
    "latitude"
].round(
    COORD_DECIMALS
)

df["longitude"] = df[
    "longitude"
].round(
    COORD_DECIMALS
)


# ==============================================================================
# 5. FEATURE ENGINEERING
# ==============================================================================

log("\n" + "=" * 80)
log("5. FEATURE ENGINEERING")
log("=" * 80)


# ------------------------------------------------------------------------------
# 5.1 Host tenure
# ------------------------------------------------------------------------------

df["host_since"] = pd.to_datetime(
    df["host_since"],
    errors="coerce"
)

df["host_tenure_days"] = (
    SNAPSHOT_DATE
    - df["host_since"]
).dt.days


df["host_tenure_days"] = (
    df["host_tenure_days"]
    .clip(
        lower=0
    )
)


# ------------------------------------------------------------------------------
# 5.2 Amenities parser
# ------------------------------------------------------------------------------

def clean_amenities_list(value):

    if pd.isna(value):
        return []

    text = str(value)

    # Remove JSON-like brackets and quotes.
    text = (
        text
        .replace("[", "")
        .replace("]", "")
        .replace("{", "")
        .replace("}", "")
        .replace('"', "")
        .replace("'", "")
    )

    # Split common delimiters.
    parts = re.split(
        r",|\||;",
        text
    )

    cleaned = []

    for item in parts:

        item = (
            item
            .strip()
            .lower()
        )

        item = re.sub(
            r"\s+",
            " ",
            item
        )

        if len(item) > 0:
            cleaned.append(item)

    return cleaned


df["amenity_tokens"] = (
    df["amenities"]
    .apply(clean_amenities_list)
)


df["amenity_count"] = (
    df["amenity_tokens"]
    .apply(len)
)


log(
    f"Mean amenities/listing: "
    f"{df['amenity_count'].mean():.2f}"
)


# ------------------------------------------------------------------------------
# 5.3 Accommodates per bedroom
# ------------------------------------------------------------------------------

df["accommodates_per_bedroom"] = (
    df["accommodates"]
    / df["bedrooms"].replace(
        0,
        np.nan
    )
)


# ------------------------------------------------------------------------------
# 5.4 Distance from city centre
#
# We initially create this using approximate city-centre coordinates where
# known. Unknown cities are handled later using training-derived city centres.
# ------------------------------------------------------------------------------

CITY_CENTERS = {

    # Singapore
    "Singapore": (
        1.290270,
        103.851959
    ),

    # London
    "London": (
        51.507351,
        -0.127758
    ),

    # Paris
    "Paris": (
        48.856613,
        2.352222
    ),

    # New York
    "New York": (
        40.712776,
        -74.005974
    ),

    # Los Angeles
    "Los Angeles": (
        34.052235,
        -118.243683
    ),

    # Rome
    "Rome": (
        41.902782,
        12.496366
    ),

    # Madrid
    "Madrid": (
        40.416775,
        -3.703790
    ),

    # Barcelona
    "Barcelona": (
        41.390205,
        2.154007
    ),

    # Lisbon
    "Lisbon": (
        38.736946,
        -9.142685
    ),

    # Sydney
    "Sydney": (
        -33.865143,
        151.209900
    )
}


def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        6371
        * 2
        * np.arcsin(
            np.sqrt(a)
        )
    )


df["dist_to_center_km"] = np.nan


for city, (
    center_lat,
    center_lon
) in CITY_CENTERS.items():

    mask = (
        df["city"].astype(str)
        == city
    )

    if mask.any():

        df.loc[
            mask,
            "dist_to_center_km"
        ] = haversine_km(
            df.loc[
                mask,
                "latitude"
            ],
            df.loc[
                mask,
                "longitude"
            ],
            center_lat,
            center_lon
        )


# For cities not in predefined centre list, calculate a centre from the data.
for city in df["city"].dropna().unique():

    mask = (
        df["city"] == city
    )

    if df.loc[
        mask,
        "dist_to_center_km"
    ].notna().any():

        continue

    city_lat = df.loc[
        mask,
        "latitude"
    ].median()

    city_lon = df.loc[
        mask,
        "longitude"
    ].median()

    if pd.notna(city_lat) and pd.notna(city_lon):

        df.loc[
            mask,
            "dist_to_center_km"
        ] = haversine_km(
            df.loc[
                mask,
                "latitude"
            ],
            df.loc[
                mask,
                "longitude"
            ],
            city_lat,
            city_lon
        )


# ==============================================================================
# 6. STRATIFIED TRAIN / TEST SPLIT
# ==============================================================================

log("\n" + "=" * 80)
log("6. STRATIFIED TRAIN / TEST SPLIT")
log("=" * 80)


# Remove cities with extremely small strata from joint city-target
# stratification if necessary.
strat_key = (
    df["city"].astype(str)
    + "_"
    + df[TARGET_COLUMN].astype(str)
)


strat_counts = strat_key.value_counts()


rare_keys = set(
    strat_counts[
        strat_counts < 2
    ].index
)


if rare_keys:

    log(
        f"Rare city-target strata (<2 rows): "
        f"{len(rare_keys)}"
    )

    # Keep the rows, but use target-only stratification if joint
    # stratification is impossible.
    use_joint_stratification = False

else:

    use_joint_stratification = True


if use_joint_stratification:

    split_stratify = strat_key

else:

    split_stratify = df[
        TARGET_COLUMN
    ]


df_train, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=split_stratify
)


df_train = df_train.copy()
df_test = df_test.copy()


log(
    f"Train rows: "
    f"{len(df_train):,}"
)

log(
    f"Test rows : "
    f"{len(df_test):,}"
)

log(
    f"Train positive rate: "
    f"{df_train[TARGET_COLUMN].mean() * 100:.2f}%"
)

log(
    f"Test positive rate : "
    f"{df_test[TARGET_COLUMN].mean() * 100:.2f}%"
)


# ==============================================================================
# 7. PREPROCESSING ARTIFACTS
# ==============================================================================

artifacts = {

    "model_metadata": {

        "task": "binary_classification",

        "target_column": TARGET_COLUMN,

        "positive_class": 1,

        "negative_class": 0,

        "snapshot_date": str(
            SNAPSHOT_DATE.date()
        ),

        "random_state": RANDOM_STATE,

        "test_size": TEST_SIZE,

        "price_cap_quantile":
            PRICE_CAP_QUANTILE,

        "count_cap_quantile":
            COUNT_CAP_QUANTILE,

        "coordinate_decimals":
            COORD_DECIMALS,

        "raw_input_encoding":
            "latin1"
    },

    "city_centers": CITY_CENTERS,

    "train_imputations": {},

    "city_price_stats": {},

    "city_price_caps": {},

    "top_amenities_vocab": [],

    "top_property_types": [],

    "candidate_feature_columns": [],

    "final_feature_columns": []
}


# ==============================================================================
# 8. TOP-K AMENITY VOCABULARY
# ==============================================================================

log("\n" + "=" * 80)
log("8. TRAIN-FITTED AMENITY VOCABULARY")
log("=" * 80)


train_amenity_tokens = []

for tokens in df_train[
    "amenity_tokens"
].apply(
    lambda x: x
):

    train_amenity_tokens.extend(
        tokens
    )


amenity_counter = Counter(
    train_amenity_tokens
)


filtered_vocab = [
    term
    for term, count
    in amenity_counter.most_common(
        TOP_K_AMENITIES
    )
    if len(term) > 2
]


artifacts[
    "top_amenities_vocab"
] = filtered_vocab


log(
    f"Amenity vocabulary size: "
    f"{len(filtered_vocab)}"
)


def add_amenity_features(
    data,
    vocabulary
):

    data = data.copy()

    for term in vocabulary:

        safe_col = (
            "amenity_"
            + re.sub(
                r"[^a-zA-Z0-9_]",
                "_",
                term.lower()
            )
            .strip("_")
        )

        data[safe_col] = (
            data[
                "amenity_tokens"
            ]
            .apply(
                lambda items:
                    int(
                        term in items
                    )
            )
        )

    return data


df_train = add_amenity_features(
    df_train,
    filtered_vocab
)

df_test = add_amenity_features(
    df_test,
    filtered_vocab
)


# ==============================================================================
# 9. TRAIN-FITTED OUTLIER CAPPING
# ==============================================================================

log("\n" + "=" * 80)
log("9. TRAIN-FITTED OUTLIER CAPPING")
log("=" * 80)


count_cap = float(
    df_train[
        "host_total_listings_count"
    ].quantile(
        COUNT_CAP_QUANTILE
    )
)


artifacts[
    "host_listings_count_cap"
] = count_cap


for data in [
    df_train,
    df_test
]:

    data[
        "host_total_listings_count"
    ] = (
        data[
            "host_total_listings_count"
        ]
        .clip(
            upper=count_cap
        )
    )


log(
    f"Host listing-count cap: "
    f"{count_cap:.2f}"
)


# ==============================================================================
# 10. WITHIN-CITY PRICE NORMALISATION
# ==============================================================================

log("\n" + "=" * 80)
log("10. WITHIN-CITY PRICE NORMALISATION")
log("=" * 80)


df_train[
    "log_price_city_zscore"
] = 0.0

df_test[
    "log_price_city_zscore"
] = 0.0


for city in (
    df_train["city"]
    .dropna()
    .unique()
):

    train_mask = (
        df_train["city"] == city
    )

    test_mask = (
        df_test["city"] == city
    )

    city_prices = df_train.loc[
        train_mask,
        "price"
    ]


    city_cap = float(
        city_prices.quantile(
            PRICE_CAP_QUANTILE
        )
    )


    artifacts[
        "city_price_caps"
    ][str(city)] = city_cap


    # Apply TRAIN-FITTED cap.
    df_train.loc[
        train_mask,
        "price"
    ] = (
        df_train.loc[
            train_mask,
            "price"
        ]
        .clip(
            upper=city_cap
        )
    )


    df_test.loc[
        test_mask,
        "price"
    ] = (
        df_test.loc[
            test_mask,
            "price"
        ]
        .clip(
            upper=city_cap
        )
    )


    log_price_train = np.log1p(
        df_train.loc[
            train_mask,
            "price"
        ]
    )


    city_mean = float(
        log_price_train.mean()
    )


    city_std = float(
        log_price_train.std()
    )


    if (
        not np.isfinite(city_std)
        or city_std <= 0
    ):

        city_std = 1.0


    artifacts[
        "city_price_stats"
    ][str(city)] = {

        "mean": city_mean,

        "std": city_std
    }


    df_train.loc[
        train_mask,
        "log_price_city_zscore"
    ] = (
        np.log1p(
            df_train.loc[
                train_mask,
                "price"
            ]
        )
        - city_mean
    ) / city_std


    df_test.loc[
        test_mask,
        "log_price_city_zscore"
    ] = (
        np.log1p(
            df_test.loc[
                test_mask,
                "price"
            ]
        )
        - city_mean
    ) / city_std


log(
    f"Cities with fitted price statistics: "
    f"{len(artifacts['city_price_stats'])}"
)


# ==============================================================================
# 11. TRAIN-FITTED IMPUTATION
# ==============================================================================

log("\n" + "=" * 80)
log("11. TRAIN-FITTED MISSING VALUE IMPUTATION")
log("=" * 80)


numeric_impute_cols = [

    "host_response_rate",

    "host_acceptance_rate",

    "host_total_listings_count",

    "bedrooms",

    "review_scores_rating",

    "review_scores_accuracy",

    "review_scores_cleanliness",

    "review_scores_checkin",

    "review_scores_communication",

    "review_scores_location",

    "review_scores_value",

    "host_tenure_days",

    "dist_to_center_km",

    "accommodates_per_bedroom"
]


for col in numeric_impute_cols:

    if col not in df_train.columns:
        continue

    # --- NEW: Create missingness indicator BEFORE imputation ---
    df_train[f"{col}_isna"] = df_train[col].isna().astype(int)
    df_test[f"{col}_isna"]  = df_test[col].isna().astype(int)
    # -----------------------------------------------------------

    median_val = float(
        df_train[col].median()
    )


    if not np.isfinite(
        median_val
    ):

        median_val = 0.0


    artifacts[
        "train_imputations"
    ][col] = median_val


    df_train[col] = (
        df_train[col]
        .fillna(median_val)
    )

    df_test[col] = (
        df_test[col]
        .fillna(median_val)
    )


categorical_impute_cols = [

    "host_response_time",

    "host_is_superhost",

    "host_has_profile_pic",

    "host_identity_verified",
    
    "city",

    "property_type",

    "room_type"
]


for col in categorical_impute_cols:

    if col not in df_train.columns:
        continue

    # --- NEW: Create missingness indicator BEFORE imputation ---
    df_train[f"{col}_isna"] = df_train[col].isna().astype(int)
    df_test[f"{col}_isna"]  = df_test[col].isna().astype(int)
    # -----------------------------------------------------------

    mode_values = (
        df_train[col]
        .mode(
            dropna=True
        )
    )


    if len(mode_values) > 0:

        mode_val = mode_values.iloc[0]

    else:

        mode_val = "Unknown"


    artifacts[
        "train_imputations"
    ][col] = str(
        mode_val
    )


    df_train[col] = (
        df_train[col]
        .fillna(mode_val)
    )

    df_test[col] = (
        df_test[col]
        .fillna(mode_val)
    )
# ==============================================================================
# 12. PROPERTY TYPE GROUPING
# ==============================================================================

log("\n" + "=" * 80)
log("12. PROPERTY TYPE GROUPING")
log("=" * 80)


top_property_types = (
    df_train[
        "property_type"
    ]
    .value_counts()
    .nlargest(10)
    .index
    .tolist()
)


artifacts[
    "top_property_types"
] = [
    str(x)
    for x in top_property_types
]


def group_property_type(value):

    if value in top_property_types:
        return value

    return "Other"


df_train[
    "property_type_grouped"
] = (
    df_train[
        "property_type"
    ]
    .apply(
        group_property_type
    )
)


df_test[
    "property_type_grouped"
] = (
    df_test[
        "property_type"
    ]
    .apply(
        group_property_type
    )
)


# ==============================================================================
# 13. EXTRACT TARGET
# ==============================================================================

y_train = (
    df_train[
        TARGET_COLUMN
    ]
    .astype(int)
    .copy()
)


y_test = (
    df_test[
        TARGET_COLUMN
    ]
    .astype(int)
    .copy()
)


# ==============================================================================
# 14. REMOVE NON-MODEL / PII / RAW TEXT COLUMNS
# ==============================================================================

log("\n" + "=" * 80)
log("14. REMOVING NON-MODEL COLUMNS")
log("=" * 80)


drop_columns = [

    # Target
    TARGET_COLUMN,

    # Identifiers / PII-like identifiers
    "listing_id",
    "host_id",

    # Raw text
    "name",
    "amenities",
    "amenity_tokens",

    # Dates
    "host_since",

    # Raw location / privacy-sensitive fields
    "host_location",
    "neighbourhood",
    "latitude",
    "longitude",
    "district",

    # Raw categorical replaced by engineered equivalent
    "property_type",

    # Raw price
    "price",

    # Intermediate field
    "log_price",

    # Any other object fields not used for modeling
    "country"
]


df_train_features = df_train.drop(
    columns=[
        c
        for c in drop_columns
        if c in df_train.columns
    ],
    errors="ignore"
)


df_test_features = df_test.drop(
    columns=[
        c
        for c in drop_columns
        if c in df_test.columns
    ],
    errors="ignore"
)


# ==============================================================================
# 15. STRICT TRAIN/TEST COLUMN ALIGNMENT
# ==============================================================================

candidate_columns = list(
    df_train_features.columns
)


df_test_features = (
    df_test_features
    .reindex(
        columns=candidate_columns,
        fill_value=0
    )
)


log(
    f"Candidate features identified: "
    f"{len(candidate_columns)}"
)


artifacts[
    "candidate_feature_columns"
] = candidate_columns


# ==============================================================================
# 16. NUMERIC CONVERSION & SECONDARY IMPUTATION
# ==============================================================================

log("\n" + "=" * 80)
log("16. FINAL PREPROCESSING ADJUSTMENTS")
log("=" * 80)

# The categorical columns required for the pipeline OHE
categorical_cols = ["city", "room_type", "property_type_grouped", "host_response_time"]

for col in candidate_columns:
    if col not in categorical_cols:
        # Convert numeric columns explicitly
        df_train_features[col] = pd.to_numeric(
            df_train_features[col],
            errors="coerce"
        )
        df_test_features[col] = pd.to_numeric(
            df_test_features[col],
            errors="coerce"
        )
        
        # Secondary imputation for engineered numeric columns
        train_median = df_train_features[col].median()
        if not np.isfinite(train_median):
            train_median = 0.0
            
        df_train_features[col] = df_train_features[col].fillna(train_median)
        df_test_features[col] = df_test_features[col].fillna(train_median)
    else:
        # Secondary imputation for categorical columns
        df_train_features[col] = df_train_features[col].fillna("Unknown")
        df_test_features[col] = df_test_features[col].fillna("Unknown")


# ==============================================================================
# 17. FINAL NaN AUDIT
# ==============================================================================

log("\n" + "=" * 80)
log("17. FINAL MODEL-READY AUDIT")
log("=" * 80)


X_train = df_train_features.copy()
X_test = df_test_features.copy()


train_nan_count = int(
    X_train.isna()
    .sum()
    .sum()
)


test_nan_count = int(
    X_test.isna()
    .sum()
    .sum()
)


log(f"X_train shape: {X_train.shape}")
log(f"X_test shape : {X_test.shape}")
log(f"y_train shape: {y_train.shape}")
log(f"y_test shape : {y_test.shape}")
log(f"Train NaN count: {train_nan_count}")
log(f"Test NaN count : {test_nan_count}")


if train_nan_count != 0:

    raise ValueError(
        f"X_train contains "
        f"{train_nan_count} NaN values."
    )

if test_nan_count != 0:

    raise ValueError(
        f"X_test contains "
        f"{test_nan_count} NaN values."
    )

if list(X_train.columns) != list(X_test.columns):

    raise ValueError(
        "Train/test feature ordering mismatch."
    )


# ==============================================================================
# 18. SAVE FINAL FEATURE SCHEMA
# ==============================================================================

log("\n" + "=" * 80)
log("18. SAVING FEATURE SCHEMA")
log("=" * 80)


FINAL_FEATURE_COLUMNS = list(
    X_train.columns
)


artifacts[
    "final_feature_columns"
] = FINAL_FEATURE_COLUMNS


feature_schema = {
    "target_column": TARGET_COLUMN,
    "positive_class": 1,
    "negative_class": 0,
    "feature_columns": FINAL_FEATURE_COLUMNS,
    "model_input_order": FINAL_FEATURE_COLUMNS,
    "categorical_columns": categorical_cols,
    "selected_feature_count": len(FINAL_FEATURE_COLUMNS)
}


schema_path = os.path.join(
    OUTPUT_DIR,
    "feature_schema.json"
)


with open(
    schema_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_schema,
        f,
        indent=2
    )


# ==============================================================================
# 19. SAVE PREPROCESSING ARTIFACTS
# ==============================================================================

artifacts_path = os.path.join(
    OUTPUT_DIR,
    "preprocessing_artifacts.json"
)


with open(
    artifacts_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifacts,
        f,
        indent=2,
        default=str
    )


# ==============================================================================
# 20. SAVE TRAIN / TEST CSV
# ==============================================================================

log("\n" + "=" * 80)
log("20. SAVING MODEL INPUT DATA")
log("=" * 80)


train_output = X_train.copy()
train_output[TARGET_COLUMN] = y_train.values

test_output = X_test.copy()
test_output[TARGET_COLUMN] = y_test.values


train_path = os.path.join(
    OUTPUT_DIR,
    "train.csv"
)

test_path = os.path.join(
    OUTPUT_DIR,
    "test.csv"
)


train_output.to_csv(
    train_path,
    index=False,
    encoding="utf-8"
)


test_output.to_csv(
    test_path,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 21. FINAL SUMMARY
# ==============================================================================

log("\n" + "=" * 80)
log("PREPROCESSING COMPLETE")
log("=" * 80)

log(f"Raw rows               : {len(df_raw):,}")
log(f"Clean rows             : {len(df):,}")
log(f"Training rows          : {len(X_train):,}")
log(f"Testing rows           : {len(X_test):,}")
log(f"Final feature count    : {X_train.shape[1]}")
log(f"Train/test schema      : {'IDENTICAL' if list(X_train.columns) == list(X_test.columns) else 'MISMATCH'}")
log(f"Train NaN count        : {train_nan_count}")
log(f"Test NaN count         : {test_nan_count}")
log(f"Train positive rate    : {y_train.mean() * 100:.2f}%")
log(f"Test positive rate     : {y_test.mean() * 100:.2f}%")
log(f"Train CSV              : {train_path}")
log(f"Test CSV               : {test_path}")
log(f"Feature schema         : {schema_path}")
log(f"Preprocessing artifacts: {artifacts_path}")

log("=" * 80)
log("READY FOR TRAINING")
log("=" * 80)

Overwriting src/preprocess.py


In [7]:
%%writefile src/train.py
import argparse
import os
import shutil
import joblib
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def log(message):
    print(message, flush=True)

def load_data():
    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    test_dir = os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test")
    
    train = pd.read_csv(os.path.join(train_dir, "train.csv"))
    test = pd.read_csv(os.path.join(test_dir, "test.csv"))
    
    target = "instant_bookable"
    y_train = train.pop(target).astype(int)
    y_test = test.pop(target).astype(int)
    
    return train, y_train, test, y_test, list(train.columns), train_dir

def evaluate(model, X, y, name):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    
    metrics = {
        f"{name}_accuracy": accuracy_score(y, predictions),
        f"{name}_precision": precision_score(y, predictions, zero_division=0),
        f"{name}_recall": recall_score(y, predictions, zero_division=0),
        f"{name}_f1": f1_score(y, predictions, zero_division=0),
        f"{name}_auc_roc": roc_auc_score(y, probabilities)
    }
    
    for k, v in metrics.items():
        log(f"{k}: {v:.4f}")
    return metrics

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=150)
    parser.add_argument("--max-depth", type=int, default=20)
    parser.add_argument("--min-samples-split", type=int, default=5)
    parser.add_argument("--min-samples-leaf", type=int, default=2)
    parser.add_argument("--random-state", type=int, default=42)
    args, _ = parser.parse_known_args()

    # 1. Load Data
    X_train, y_train, X_test, y_test, feature_names, train_dir = load_data()

    # 2. Define Feature Types explicitly for the Pipeline
    categorical_cols = ["city", "room_type", "property_type_grouped", "host_response_time"]
    numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

    # 3. Build the ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
        ]
    )

    # 4. Build the Pipeline
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            min_samples_split=args.min_samples_split,
            min_samples_leaf=args.min_samples_leaf,
            random_state=args.random_state
        ))
    ])

    # 5. Train the bundled Pipeline
    log("Training unified Pipeline (OHE + Scaler + RandomForest)...")
    model_pipeline.fit(X_train, y_train)
    log("✓ Pipeline training completed.")

    # 6. Evaluation
    metrics = {}
    metrics.update(evaluate(model_pipeline, X_train, y_train, "train"))
    metrics.update(evaluate(model_pipeline, X_test, y_test, "test"))

    # 7. Save the bundled Pipeline to model.joblib
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(model_pipeline, os.path.join(model_dir, "model.joblib"))
    log("✓ Model Pipeline saved successfully.")
    
    # 8. Copy JSON Artifacts into the model directory so they are bundled into model.tar.gz
    log("Copying inference artifacts to model directory...")
    shutil.copy(os.path.join(train_dir, "feature_schema.json"), os.path.join(model_dir, "feature_schema.json"))
    shutil.copy(os.path.join(train_dir, "preprocessing_artifacts.json"), os.path.join(model_dir, "preprocessing_artifacts.json"))
    log("✓ Artifacts successfully bundled for endpoint deployment.")

Overwriting src/train.py


In [8]:
# No MLflow requirements file is needed in the SageMaker training container.
# MLflow logging is done after the pipeline completes, from this notebook.

print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (32374 bytes)
  src/train.py  (4065 bytes)
  src/inference.py  (17134 bytes)


## 1A. Upload pipeline source files to S3

This section pre-places the three pipeline source files in the team S3 area.

Files uploaded:

preprocess.py
train.py
inference.py
There is no requirements_train.txt: the SageMaker training container does not need MLflow or MLflow App packages. MLflow logging happens later from this notebook to the SageMaker MLflow App.

S3 location example:

s3://nyp-26s1-iti113/iti113/team40/data/heart-disease/pipeline_src/

In [9]:
%%writefile src/inference.py

import os
import json
import re
import pickle
import traceback

import numpy as np
import pandas as pd


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_DIR = os.environ.get(
    "SM_MODEL_DIR",
    "/opt/ml/model"
)

print("=" * 80)
print("AIRBNB CLASSIFICATION INFERENCE")
print("=" * 80)
print(f"Model directory: {MODEL_DIR}")


# =============================================================================
# MODEL LOADING
# =============================================================================

def model_fn(model_dir):

    print("=" * 80)
    print("LOADING MODEL PIPELINE")
    print("=" * 80)

    possible_model_files = [
        "model.joblib",
        "model.pkl"
    ]

    model_path = None

    for filename in possible_model_files:

        candidate = os.path.join(
            model_dir,
            filename
        )

        if os.path.exists(candidate):

            model_path = candidate
            break

    if model_path is None:
        raise FileNotFoundError(
            "Could not find model.joblib or model.pkl "
            f"in {model_dir}"
        )

    print(f"Loading model from: {model_path}")

    if model_path.endswith(".joblib"):
        import joblib
        model = joblib.load(model_path)
    else:
        with open(model_path, "rb") as f:
            model = pickle.load(f)

    print(f"Model Pipeline loaded: {type(model).__name__}")

    # -------------------------------------------------------------------------
    # Load feature schema
    # -------------------------------------------------------------------------

    schema_path = os.path.join(
        model_dir,
        "feature_schema.json"
    )

    if not os.path.exists(schema_path):
        raise FileNotFoundError(
            "feature_schema.json was not found in "
            f"{model_dir}"
        )

    with open(schema_path, "r") as f:
        feature_schema = json.load(f)

    feature_columns = feature_schema.get("feature_columns", [])
    print(f"Feature schema loaded: {len(feature_columns)} features expected before pipeline transformation.")

    # -------------------------------------------------------------------------
    # Load preprocessing artifacts
    # -------------------------------------------------------------------------

    artifacts_path = os.path.join(
        model_dir,
        "preprocessing_artifacts.json"
    )

    if not os.path.exists(artifacts_path):
        raise FileNotFoundError(
            "preprocessing_artifacts.json was not found "
            f"in {model_dir}"
        )

    with open(artifacts_path, "r") as f:
        artifacts = json.load(f)

    print("Preprocessing artifacts loaded.")
    print("=" * 80)

    return {
        "model": model,
        "feature_schema": feature_schema,
        "artifacts": artifacts
    }


# =============================================================================
# INPUT PARSING
# =============================================================================

def input_fn(request_body, request_content_type):

    print("=" * 80)
    print("PARSING REQUEST")
    print("=" * 80)
    print(f"Content type: {request_content_type}")

    if isinstance(request_body, bytes):
        request_body = request_body.decode("utf-8")

    if request_content_type == "application/json":
        data = json.loads(request_body)
        return data

    if request_content_type in ("application/jsonlines", "application/jsonl"):
        return [
            json.loads(line)
            for line in request_body.strip().splitlines()
            if line.strip()
        ]

    if request_content_type == "text/csv":
        return pd.read_csv(pd.io.common.StringIO(request_body))

    raise ValueError(f"Unsupported content type: {request_content_type}")


# =============================================================================
# AMENITY CLEANING
# =============================================================================

def clean_amenities_list(value):

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    text = text.strip("{}[]")
    tokens = []

    for token in text.split(","):
        token = token.strip().strip("\"'")
        token = re.sub(r"\s+", " ", token).strip().lower()
        if token:
            tokens.append(token)

    return tokens


# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

def create_features(df, artifacts, feature_schema):

    print("=" * 80)
    print("CREATING INFERENCE FEATURES")
    print("=" * 80)

    df = df.copy()

    # =========================================================================
    # ARTIFACTS
    # =========================================================================

    train_imputations = artifacts.get("train_imputations", {})
    city_price_stats = artifacts.get("city_price_stats", {})
    city_price_caps = artifacts.get("city_price_caps", {})
    top_amenities_vocab = artifacts.get("top_amenities_vocab", [])
    top_property_types = artifacts.get("top_property_types", [])
    
    final_feature_columns = feature_schema.get("feature_columns", [])
    categorical_cols = feature_schema.get("categorical_columns", [])

    # =========================================================================
    # BASIC VALIDATION
    # =========================================================================

    required_raw_columns = [
        "city",
        "amenities",
        "host_total_listings_count",
        "price"
    ]

    missing_required = [col for col in required_raw_columns if col not in df.columns]

    if missing_required:
        raise ValueError(
            "Missing required inference fields: " + ", ".join(missing_required)
        )

    # =========================================================================
    # 1. AMENITIES
    # =========================================================================
    print("Creating amenity features...")
    parsed_amenities = df["amenities"].apply(clean_amenities_list)

    for term in top_amenities_vocab:
        safe_col = "amenity_" + re.sub(r"[^a-zA-Z0-9_]", "_", str(term).lower()).strip("_")
        df[safe_col] = parsed_amenities.apply(lambda items: int(str(term).lower() in items))

    # =========================================================================
    # 2. HOST LISTING COUNT CAPPING
    # =========================================================================
    count_cap = artifacts.get("host_listings_count_cap")
    if count_cap is not None:
        df["host_total_listings_count"] = (
            pd.to_numeric(df["host_total_listings_count"], errors="coerce")
            .clip(upper=float(count_cap))
        )

    # =========================================================================
    # 3. WITHIN-CITY PRICE NORMALIZATION
    # =========================================================================
    print("Creating within-city price feature...")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["log_price_city_zscore"] = 0.0

    for city in df["city"].dropna().unique():
        city_key = str(city)
        mask = (df["city"].astype(str) == city_key)
        
        stats = city_price_stats.get(city_key)
        cap = city_price_caps.get(city_key)

        if stats is None:
            df.loc[mask, "log_price_city_zscore"] = 0.0
            continue

        price_values = df.loc[mask, "price"]
        if cap is not None:
            price_values = price_values.clip(upper=float(cap))

        log_price = np.log1p(price_values)
        mean_value = float(stats.get("mean", 0.0))
        std_value = float(stats.get("std", 1.0))

        if std_value <= 0:
            std_value = 1.0

        df.loc[mask, "log_price_city_zscore"] = (log_price - mean_value) / std_value
        
    # =========================================================================
    # 4. TRAINING-FITTED IMPUTATION
    # =========================================================================
    print("Applying training-fitted imputations...")
    for col, value in train_imputations.items():
        if col not in df.columns:
            continue

        # --- NEW: Create missingness indicator during inference ---
        df[f"{col}_isna"] = df[col].isna().astype(int)
        # ----------------------------------------------------------

        if isinstance(value, (int, float)):
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(float(value))
        else:
            df[col] = df[col].fillna(value)

    # =========================================================================
    # 5. DERIVED FEATURES
    # =========================================================================
    if "host_since" in df.columns and "host_tenure_days" not in df.columns:
        host_date = pd.to_datetime(df["host_since"], errors="coerce")
        snapshot_date = artifacts.get("model_metadata", {}).get("snapshot_date")
        
        if snapshot_date:
            snapshot = pd.Timestamp(snapshot_date)
            df["host_tenure_days"] = (snapshot - host_date).dt.days

    if "accommodates" in df.columns and "bedrooms" in df.columns:
        accommodates = pd.to_numeric(df["accommodates"], errors="coerce")
        bedrooms = pd.to_numeric(df["bedrooms"], errors="coerce")
        df["accommodates_per_bedroom"] = accommodates / bedrooms.replace(0, np.nan)

    if "amenity_count" not in df.columns:
        df["amenity_count"] = parsed_amenities.apply(len)

    # =========================================================================
    # 6. PROPERTY TYPE GROUPING
    # =========================================================================
    if "property_type" in df.columns:
        df["property_type_grouped"] = df["property_type"].apply(
            lambda value: value if value in top_property_types else "Other"
        )

    # =========================================================================
    # 7. DROP NON-MODEL COLUMNS
    # =========================================================================
    drop_unneeded = [
        "instant_bookable", "amenities", "host_since", "host_location",
        "neighbourhood", "property_type", "price", "log_price"
    ]
    df = df.drop(columns=[col for col in drop_unneeded if col in df.columns], errors="ignore")

    # =========================================================================
    # 8. STRICT FEATURE SCHEMA ALIGNMENT
    # =========================================================================
    print("Aligning to training feature schema...")
    df = df.reindex(columns=final_feature_columns, fill_value=0)
    print(f"Features after schema alignment: {df.shape[1]}")

    # =========================================================================
    # 9. NUMERIC CONVERSION
    # =========================================================================
    # Only convert numeric columns. Leave categorical text strings untouched for the Pipeline.
    for col in df.columns:
        if col not in categorical_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            
            # Safety fallback imputation for dynamically engineered columns
            train_median = artifacts.get("train_imputations", {}).get(col, 0.0)
            df[col] = df[col].fillna(train_median)
        else:
            # Fallback for categorical NaNs introduced during alignment
            df[col] = df[col].fillna("Unknown")

    # =========================================================================
    # 10. FINAL CONTRACT CHECK
    # =========================================================================
    if list(df.columns) != list(final_feature_columns):
        raise ValueError("Final inference feature ordering does not match training schema.")

    if int(df.isna().sum().sum()) != 0:
        nan_columns = df.columns[df.isna().any()].tolist()
        raise ValueError(f"NaN values remain after preprocessing. Columns: {nan_columns}")

    print(f"FINAL FEATURE COUNT (Pre-Pipeline): {df.shape[1]}")
    return df


# =============================================================================
# PREDICTION IMPLEMENTATION
# =============================================================================

def _predict_fn(input_data, model_bundle):

    print("=" * 80)
    print("RUNNING PREDICTION")
    print("=" * 80)

    model = model_bundle["model"]
    feature_schema = model_bundle["feature_schema"]
    artifacts = model_bundle["artifacts"]

    # =========================================================================
    # Convert input into DataFrame
    # =========================================================================
    if isinstance(input_data, pd.DataFrame):
        df = input_data.copy()
    elif isinstance(input_data, dict):
        if "data" in input_data and isinstance(input_data["data"], dict):
            df = pd.DataFrame([input_data["data"]])
        elif "data" in input_data and isinstance(input_data["data"], list):
            df = pd.DataFrame(input_data["data"])
        else:
            df = pd.DataFrame([input_data])
    elif isinstance(input_data, list):
        df = pd.DataFrame(input_data)
    else:
        raise ValueError(f"Unsupported input data type: {type(input_data)}")

    print(f"Incoming rows: {len(df)}")

    # =========================================================================
    # Feature transformation
    # =========================================================================
    X = create_features(df, artifacts, feature_schema)
    print(f"Prediction matrix shape (before Pipeline OHE/Scaling): {X.shape}")

    # =========================================================================
    # Prediction (Pipeline handles OHE and Scaling automatically)
    # =========================================================================
    prediction = model.predict(X)

    # =========================================================================
    # Probability
    # =========================================================================
    probability_class_1 = np.full(len(X), np.nan)

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        
        # In a Pipeline, the classes_ attribute belongs to the final estimator
        classifier = model.named_steps['classifier']
        classes = list(classifier.classes_)
        print(f"Model classes: {classes}")

        if 1 in classes:
            positive_index = classes.index(1)
            probability_class_1 = probabilities[:, positive_index]
        elif "1" in classes:
            positive_index = classes.index("1")
            probability_class_1 = probabilities[:, positive_index]
        else:
            print(f"WARNING: Class 1 was not found in model classes {classes}.")
            print("Returning NaN for probability_class_1.")
    else:
        print("WARNING: Model does not provide predict_proba().")

    # =========================================================================
    # Build response
    # =========================================================================
    results = []
    for i in range(len(X)):
        pred = prediction[i]
        if isinstance(pred, np.generic):
            pred = pred.item()

        prob = probability_class_1[i]
        if isinstance(prob, np.generic):
            prob = prob.item()

        result = {
            "prediction": pred,
            "probability_class_1": None if pd.isna(prob) else float(prob),
            "class_1": 1,
            "class_0": 0
        }
        results.append(result)

    print("Prediction complete.")
    return results


# =============================================================================
# SAGEMAKER PREDICTION FUNCTION
# =============================================================================

def predict_fn(input_data, model_bundle):
    try:
        return _predict_fn(input_data, model_bundle)
    except Exception as e:
        print("=" * 80)
        print("INFERENCE ERROR")
        print("=" * 80)
        print(f"Exception type: {type(e).__name__}")
        print(f"Exception message: {str(e)}")
        traceback.print_exc()
        raise


# =============================================================================
# OUTPUT SERIALIZATION
# =============================================================================

def output_fn(prediction, accept):
    print("=" * 80)
    print("SERIALIZING RESPONSE")
    print("=" * 80)

    if accept in ("application/json", "*/*", None):
        return (json.dumps(prediction, allow_nan=False), "application/json")

    raise ValueError(f"Unsupported response content type: {accept}")

Overwriting src/inference.py


In [10]:
from pathlib import Path

s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/preprocess.py


Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/train.py


Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src


## 1B. Download pipeline source files from S3
The SageMaker Pipeline below will use the local pipeline_src/ folder, but that folder is recreated by downloading the source files from S3.

This verifies that the pipeline is using the S3-preplaced source files rather than directly depending on the original notebook-generated src/ files.

In [11]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/train.py -> pipeline_src/train.py
Downloaded s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the SageMaker Pipeline
Four SageMaker Pipeline steps:

1. ProcessingStep — runs preprocess.py, outputs train/test CSVs to S3.
2. TrainingStep — runs train.py, prints metrics for SageMaker and saves the model artefact.
3. ConditionStep — checks AUC ≥ threshold before allowing registration.
4. ModelStep — registers the model in SageMaker Model Registry (PendingManualApproval).
MLflow is not inside the pipeline. A later notebook section logs the completed SageMaker run into the team SageMaker MLflow App experiment.

In [12]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

from sagemaker.workflow.parameters import (
    ParameterFloat,
    ParameterInteger,
    ParameterString,
)

pipeline_session = PipelineSession()

# -------------------------------------------------------------------------
# Pipeline parameters — can be overridden at execution time
# -------------------------------------------------------------------------
p_n_est = ParameterInteger(name="NEstimators", default_value=150)
p_depth = ParameterInteger(name="MaxDepth", default_value=20)
p_features = ParameterString(name="MaxFeatures", default_value="sqrt")
p_min_leaf = ParameterInteger(name="MinSamplesLeaf", default_value=2)
p_min_split = ParameterInteger(name="MinSamplesSplit", default_value=5)
p_n_jobs = ParameterInteger(name="NJobs", default_value=2)
p_rand_state = ParameterInteger(name="RandomState", default_value=42)

# Evaluation metric threshold (optional)
p_gate = ParameterFloat(name="QualityGateAUC", default_value=QUALITY_GATE_AUC)

print("Pipeline parameters defined.")

Pipeline parameters defined.


In [13]:
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep

# ==============================================================================
# Step 1: ProcessingStep (Data Cleaning & Feature Engineering)
# ==============================================================================

# 1. Initialize the Scikit-learn Processor
processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-process",
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

# 2. Define the ProcessingStep
step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=RAW_DATA_URI,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="processed",
            source="/opt/ml/processing/output",
            destination=f"{PIPELINE_ROOT}/processed",
        )
    ],
    code=f"{LOCAL_PIPELINE_SRC}/preprocess.py",
    job_arguments=[
        "--test-size", "0.2",
        "--random-state", "42",
    ],
)

print("Step 1 (ProcessingStep) defined.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [14]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

# ==============================================================================
# Step 2: TrainingStep (Tuned Random Forest Model)
# ==============================================================================
metric_definitions = [
    {
        "Name": "test_accuracy",
        "Regex": r"test_accuracy:\s*([0-9.]+)"
    },
    {
        "Name": "test_precision",
        "Regex": r"test_precision:\s*([0-9.]+)"
    },
    {
        "Name": "test_recall",
        "Regex": r"test_recall:\s*([0-9.]+)"
    },
    {
        "Name": "test_f1",
        "Regex": r"test_f1:\s*([0-9.]+)"
    },
    {
        "Name": "test_auc_roc",
        "Regex": r"test_auc_roc:\s*([0-9.]+)"
    }
]
# Estimator definition using the Scikit-learn container
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "max-features": p_features,
        "min-samples-leaf": p_min_leaf,
        "min-samples-split": p_min_split,
        "n-jobs": p_n_jobs,
        "random-state": p_rand_state,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    metric_definitions=metric_definitions,
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

# Reference the preprocessed dataset S3 URI from Step 1 (ProcessingStep)
processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

# Define the SageMaker Pipeline TrainingStep
step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
        "test": TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
    },
)

print("Step 2 (TrainingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [15]:
# Step 3: ModelStep — register in SageMaker Model Registry
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [16]:
# Step 4: ConditionStep — gate on SageMaker-captured test AUC
#
# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")

Step 4 (ConditionStep) defined.


In [17]:
# ==============================================================================
# Assemble and Upsert the SageMaker Pipeline
# ==============================================================================

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        p_n_est,
        p_depth,
        p_features,
        p_min_leaf,
        p_min_split,
        p_n_jobs,
        p_rand_state,
        p_gate,
    ],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)

print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print("View in SageMaker Studio: left sidebar -> Pipelines")

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team14-airbnb-instant-booking" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 3. Execute the Pipeline

In [18]:
import time

# -------------------------------------------------------------------------
# Start Pipeline Execution with All Tuned Hyperparameters
# -------------------------------------------------------------------------
execution = pipeline.start(
    parameters={
        "NEstimators": 150,
        "MaxDepth": 20,
        "MaxFeatures": "sqrt",
        "MinSamplesLeaf": 2,
        "MinSamplesSplit": 5,
        "NJobs": 2,
        "RandomState": 42,
        "QualityGateAUC": 0.75,
    }
)

print(f"Execution ARN: {execution.arn}")
print("Monitoring step status below. Takes ~10-15 minutes.\n")

# Wait for the pipeline execution to complete
execution.wait()

# Display step status summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION STEP SUMMARY")
print("=" * 60)

steps = execution.list_steps()
for step in reversed(steps):
    step_name = step.get("StepName")
    status = step.get("StepStatus")
    print(f"  • {step_name:<25}: {status}")

execution_status = execution.describe()["PipelineExecutionStatus"]
print(f"\nFinal Pipeline Status: {execution_status}")
print("=" * 60)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team14-airbnb-instant-booking/execution/059euzj1rjtu
Monitoring step status below. Takes ~10-15 minutes.




PIPELINE EXECUTION STEP SUMMARY
  • PreprocessData           : Succeeded
  • TrainModel               : Succeeded
  • AUCQualityGate           : Succeeded
  • RegisterModel-RepackModel-0: Succeeded
  • RegisterModel-RegisterModel: Succeeded

Final Pipeline Status: Succeeded


In [19]:
## DEBUG stuck stage
print("=" * 70)
print("PIPELINE EXECUTION STATUS")
print("=" * 70)

for step in execution.list_steps():
    print(f"\nSTEP: {step['StepName']}")
    print(f"STATUS: {step['StepStatus']}")

    if step.get("FailureReason"):
        print("FAILURE DETAILS:")
        print(step["FailureReason"])

PIPELINE EXECUTION STATUS



STEP: RegisterModel-RegisterModel
STATUS: Succeeded

STEP: RegisterModel-RepackModel-0
STATUS: Succeeded

STEP: AUCQualityGate
STATUS: Succeeded

STEP: TrainModel
STATUS: Succeeded

STEP: PreprocessData
STATUS: Succeeded


In [20]:
steps = execution.list_steps()

for step in steps:
    print("=" * 70)
    print("STEP NAME:", step["StepName"])
    print("STATUS:", step["StepStatus"])

    metadata = step.get("Metadata", {})
    print("METADATA:")
    print(metadata)

STEP NAME: RegisterModel-RegisterModel
STATUS: Succeeded
METADATA:
{'RegisterModel': {'Arn': 'arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team14-airbnb-instant-booking/29'}}
STEP NAME: RegisterModel-RepackModel-0
STATUS: Succeeded
METADATA:
{'TrainingJob': {'Arn': 'arn:aws:sagemaker:ap-southeast-1:044528205969:training-job/pipelines-059euzj1rjtu-RegisterModel-Repack-EqaR9hFOqW'}}
STEP NAME: AUCQualityGate
STATUS: Succeeded
METADATA:
{'Condition': {'Outcome': 'True'}}
STEP NAME: TrainModel
STATUS: Succeeded
METADATA:
{'TrainingJob': {'Arn': 'arn:aws:sagemaker:ap-southeast-1:044528205969:training-job/pipelines-059euzj1rjtu-TrainModel-RiTljuI86P'}}
STEP NAME: PreprocessData
STATUS: Succeeded
METADATA:
{'ProcessingJob': {'Arn': 'arn:aws:sagemaker:ap-southeast-1:044528205969:processing-job/pipelines-059euzj1rjtu-PreprocessData-i1wozo1ryX'}}


In [21]:
import time

prev = {}

while True:

    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()

    # Compatible with both old and new SageMaker SDKs
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]

        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        break

    time.sleep(30)

RegisterModel-RegisterModel Succeeded
RegisterModel-RepackModel-0 Succeeded
AUCQualityGate       Succeeded
TrainModel           Succeeded
PreprocessData       Succeeded

Pipeline Status: Succeeded


## 4. Log the Completed SageMaker Run to the SageMaker MLflow App
Run this section only after the pipeline has succeeded.

This notebook-side step reads the completed SageMaker Training Job to obtain:

- captured quality metrics,
- training hyperparameters,
- training-job name and pipeline execution ARN,
- SageMaker model-artifact S3 URI.
It then logs these as an MLflow run in the team experiment hosted by the SageMaker Serverless MLflow App created in Notebook 01. No Databricks host or token is required.

In [22]:
# Run only after the execution-monitoring cell reports "Pipeline Succeeded".
# In SageMaker SDK 2.257.3, execution.list_steps() returns a Python list.
# The compatibility helper also supports SDK versions that return a dictionary.
import mlflow


def get_pipeline_steps(execution):
    response = execution.list_steps()
    if isinstance(response, list):
        return response
    return response.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError(
        "The SageMaker Pipeline has not succeeded. "
        "Resolve pipeline failures before logging to MLflow."
    )

steps = get_pipeline_steps(execution)

print("Pipeline steps:")
for step in steps:
    print(f"  {step['StepName']}: {step['StepStatus']}")

train_step_info = next(
    (
        step for step in steps
        if step["StepName"] == "TrainModel"
        and step["StepStatus"] == "Succeeded"
    ),
    None
)

if train_step_info is None:
    raise RuntimeError(
        "A successful TrainModel step was not found in this pipeline execution."
    )

training_job_arn = train_step_info["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
training_job = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

# SageMaker captures the metrics printed by train.py through metric_definitions.
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"] in {"test_auc_roc", "test_accuracy", "test_f1"}
}

if not captured_metrics:
    raise RuntimeError(
        "No captured SageMaker metrics were found. "
        "Check train.py output and estimator.metric_definitions."
    )

model_artifact_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
training_hyperparameters = training_job.get("HyperParameters", {})

print("Training job:", training_job_name)
print("Model artefact:", model_artifact_s3_uri)
print("Captured metrics:", captured_metrics)

# Log to SageMaker Serverless MLflow App.
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow_run_name = (
    f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_"
    f"{int(time.time())}"
)

with mlflow.start_run(run_name=mlflow_run_name) as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "heart-disease",
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_s3_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    })

    # Hyperparameters arrive from SageMaker as strings, which are valid MLflow params.
    mlflow.log_params(training_hyperparameters)
    mlflow.log_metrics(captured_metrics)

    # Store a small, portable traceability record as an MLflow artefact.
    run_summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_s3_uri,
        "metrics": captured_metrics,
        "hyperparameters": training_hyperparameters,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "semester": SEMESTER,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
    }

    summary_file = "sagemaker_pipeline_run_summary.json"
    with open(summary_file, "w") as f:
        json.dump(run_summary, f, indent=2)

    mlflow.log_artifact(
        summary_file,
        artifact_path="sagemaker_pipeline"
    )

    mlflow_run_id = run.info.run_id
    mlflow_experiment_id = run.info.experiment_id

print("SageMaker MLflow App logging completed.")
print("MLflow run ID:", mlflow_run_id)
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", mlflow_experiment_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=mlflow_experiment_id,
    run_id=mlflow_run_id
)

Pipeline steps:
  RegisterModel-RegisterModel: Succeeded
  RegisterModel-RepackModel-0: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded


Training job: pipelines-059euzj1rjtu-TrainModel-RiTljuI86P
Model artefact: s3://sagemaker-ap-southeast-1-044528205969/pipelines-059euzj1rjtu-TrainModel-RiTljuI86P/output/model.tar.gz
Captured metrics: {'test_auc_roc': 0.8360999822616577, 'test_accuracy': 0.7627999782562256, 'test_f1': 0.6812999844551086}


🏃 View run team14_s1401_sagemaker_pipeline_1787331934 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2/runs/7e5eb44f5e314ebd94683153d614891b
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2
SageMaker MLflow App logging completed.
MLflow run ID: 7e5eb44f5e314ebd94683153d614891b
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS
Experiment: ITI113/team14/InstantBooking-Classification
Experiment ID: 2

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlFDWUlBUCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGFMY1k2dXBOZDlKWFBRRVRENnZXRVJOL3YybDRxS3RTY3NZWlRuTlVsNHdBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGb1dDOTROMFZQVkdObFZVdHNSM0JZZDFSaVluQllUamwwYW1kWVUxbHdZa1V6T0VVNEwwWlRUR3h0Y0hkaVVtUXlkbFo1SzBOT2N6UTBWWEJLTlZncmR6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFWRjNuZEhXQUZLUVgrSGh3OWhmTmk4QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF5MlYvWnlmLzdxUkpyNFRWUUNBUkNBTzI3azBVamJIaDhtWVNxM29WTkt4Z1R0TWh3TElib0ZVdVpkUGZ4STRaTXpKc3hvNk1XWXBYTURHdEVGOTB6SDJ1eU1kRENhbG1QTFlERFdBZ0FBRUFEUWw5ck5Kb3Z2ZnR6NlNkOUxmWDBhbkI1NlZlOSs1QjM4MVhhcHBOMFlzUG9xSmdrdDA4VU


Presigned MLflow run URL:
https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlhGRjVLUyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDJoMVd2dmo2L0RSZUtmVjg3Vk93Y2dOS29iL3UvanV6aHlWTHAzbVJCU0FBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGdkwwVTNlU3RqYTJaU05FcHpla2wwYkZKdGJFSkhlVEZYZW1kbU9HdEdWWFJNTjFwMk5UZ3diVFV2WVVob00yeFdaVGhOWW5oMVdEVmtRVlpzVVZBMlFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFmL0E1ZWRNdnl0L2FCUTdLVGZLYmhnQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6cUFLYnlURktZRTNLd3JIUUNBUkNBTzFYZ2JsK0FSNVpGUldoUFlLNjZjK1hheHdUcHVDSnZGOGFzNElXTk51ZHlxY1pHQVh4c2FnSmRVU2lzUGFPNGdva2crRVczalp0WDBud2FBZ0FBRUFCMk54ZmIzemN0YXdDOW5uZUhCcTNTM25DVG8wdjcxM0JNQUpoOXVSSkhiRDdFUjNwZXNReEtFSDg2

## 5. Deploy Serverless Endpoint
After the pipeline succeeds, the model sits in Model Registry with PendingManualApproval. We approve it here, then deploy as a Serverless Endpoint — cost is near-zero when idle.

In [23]:
sm = boto3.client('sagemaker')

# Get the latest registered model package
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline completed the Register step.')
else:
    pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package : {pkg_arn}')
    print(f'Status        : {pkgs[0]["ModelApprovalStatus"]}')

Model package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team14-airbnb-instant-booking/29
Status        : PendingManualApproval


In [24]:
# Approve the model
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {pkg_arn}')

Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team14-airbnb-instant-booking/29


In [25]:
import sagemaker
import botocore
import time

sess = sagemaker.Session()
sm_client = sess.sagemaker_client

ENDPOINT_NAME = "iti113-team14-airbnb-instant-booking"


# ============================================================
# DELETE EXISTING ENDPOINT
# ============================================================

try:
    sm_client.delete_endpoint(
        EndpointName=ENDPOINT_NAME
    )
    print(f"Deleted endpoint: {ENDPOINT_NAME}")

except botocore.exceptions.ClientError as e:
    if "Could not find" in str(e):
        print("Endpoint does not exist.")
    else:
        raise


# ============================================================
# WAIT FOR ENDPOINT TO BE DELETED
# ============================================================

print("Waiting for endpoint deletion...")

while True:

    try:
        response = sm_client.describe_endpoint(
            EndpointName=ENDPOINT_NAME
        )

        status = response["EndpointStatus"]

        print(f"Endpoint status: {status}")

        time.sleep(10)

    except botocore.exceptions.ClientError as e:

        if "Could not find" in str(e):
            print("Endpoint completely deleted.")
            break

        raise


# ============================================================
# DELETE OLD ENDPOINT CONFIG
# ============================================================

try:
    sm_client.delete_endpoint_config(
        EndpointConfigName=ENDPOINT_NAME
    )

    print(
        f"Deleted endpoint config: {ENDPOINT_NAME}"
    )

except botocore.exceptions.ClientError as e:

    if "Could not find" in str(e):
        print("Endpoint config does not exist.")

    else:
        raise

Deleted endpoint: iti113-team14-airbnb-instant-booking
Waiting for endpoint deletion...


Endpoint status: Deleting


Endpoint completely deleted.
Deleted endpoint config: iti113-team14-airbnb-instant-booking


In [26]:
from sagemaker import ModelPackage
from sagemaker.serverless import ServerlessInferenceConfig

deployable = ModelPackage(
    role=role,
    model_package_arn=pkg_arn,
    sagemaker_session=sess
)

serverless_cfg = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

print(
    f"Deploying serverless endpoint: "
    f"{ENDPOINT_NAME}"
)

predictor = deployable.deploy(
    serverless_inference_config=serverless_cfg,
    endpoint_name=ENDPOINT_NAME
)

print(
    f"Endpoint ready: {ENDPOINT_NAME}"
)

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: ModelPackage is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Deploying serverless endpoint: iti113-team14-airbnb-instant-booking


INFO:sagemaker:Creating model with name: team14-airbnb-instant-booking-2026-08-21-17-05-47-483


INFO:sagemaker:Creating endpoint-config with name iti113-team14-airbnb-instant-booking


INFO:sagemaker:Creating endpoint with name iti113-team14-airbnb-instant-booking


...

-

-

-

!

Endpoint ready: iti113-team14-airbnb-instant-booking


In [27]:
import boto3
import json

# 1. Initialize the SageMaker Runtime client
sm_runtime = boto3.client("sagemaker-runtime", region_name="ap-southeast-1")

# 2. Define your endpoint name (matching your notebook's configuration)
ENDPOINT_NAME = "iti113-team14-airbnb-instant-booking"

# 3. Define the raw test payload
test_listing = {
  "city": "Singapore",
  "amenities": "[\"Wifi\", \"Air conditioning\", \"Kitchen\", \"Pool\", \"Gym\", \"TV\"]",
  "host_total_listings_count": 2,
  "price": 150.0,
  "host_since": "2018-05-15",
  "accommodates": 4,
  "bedrooms": 2,
  "property_type": "Entire apartment",
  "room_type": "Entire home/apt",
  "host_response_time": "within an hour",
  "host_response_rate": 1.0,
  "host_acceptance_rate": 0.95,
  "host_is_superhost": 1,
  "host_has_profile_pic": 1,
  "host_identity_verified": 1,
  "minimum_nights": 2,
  "maximum_nights": 1125,
  "review_scores_rating": 4.8,
  "review_scores_accuracy": 4.9,
  "review_scores_cleanliness": 4.8,
  "review_scores_checkin": 5.0,
  "review_scores_communication": 5.0,
  "review_scores_location": 4.9,
  "review_scores_value": 4.7,
  "latitude": 1.290,
  "longitude": 103.851
}

print(f"Sending raw profile to endpoint: {ENDPOINT_NAME}...\n")

# 4. Invoke the endpoint
response = sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(test_listing)
)

# 5. Parse and print the response
response_body = response["Body"].read().decode("utf-8")
prediction_result = json.loads(response_body)

print("Endpoint Response:")
print(json.dumps(prediction_result, indent=2))

Sending raw profile to endpoint: iti113-team14-airbnb-instant-booking...



Endpoint Response:
[
  {
    "prediction": 1,
    "probability_class_1": 0.5232764955677277,
    "class_1": 1,
    "class_0": 0
  }
]


## 6. Test the Live Endpoint

## Delete endpoint when no lonnger needed (below code)

In [28]:
# # To delete endpoint when no longer needed, uncomment:
# boto3.client('sagemaker').delete_endpoint(EndpointName=ENDPOINT_NAME)
# print(f'Endpoint deleted: {ENDPOINT_NAME}')

In [29]:
# print('=' * 55)
# print('NOTEBOOK 03 COMPLETE')
# print('=' * 55)
# print(f'Pipeline : {PIPELINE_NAME}')
# print(f'MLflow   : {MLFLOW_EXPERIMENT_NAME} on SageMaker MLflow App')
# print(f'MLflow App ARN : {MLFLOW_APP_ARN}')
# print(f'SageMaker Registry : {MODEL_PACKAGE_GROUP}')
# print(f'Endpoint : {ENDPOINT_NAME} (Serverless)')
# print()
# print('Next: Notebook 04 — AI Governance, Bias & Explainability')

## Checklist before Notebook 04

[] src/preprocess.py, train.py, inference.py written and reviewed
[] Pipeline upserted (visible in SageMaker Studio under Pipelines)
[] Pipeline execution completed (all steps green)
[] Post-pipeline MLflow run visible in the team SageMaker MLflow App experiment
[] Model registered in SageMaker Model Registry after passing AUC gate
[] Serverless Endpoint deployed and responding to test calls
[] Both high-risk and low-risk test profiles return sensible predictions

## Below Code Gets the model from endpoint (make sure endpoint is not deleted)

In [ ]:
import os
import json
import boto3
from urllib.parse import urlparse
from pathlib import Path
from botocore.exceptions import ClientError


# ============================================================
# CONFIGURATION
# ============================================================

REGION = "ap-southeast-1"

# Your deployed serverless endpoint name
ENDPOINT_NAME = ENDPOINT_NAME #"REPLACE_WITH_YOUR_ENDPOINT_NAME"

# Your class/team bucket and desired destination folder
DESTINATION_BUCKET = "nyp-26s1-iti113" # "REPLACE_WITH_YOUR_CLASS_BUCKET"

# Recommended destination naming
DESTINATION_KEY = (
    f"iti113/{TEAM_ID}/models/"
    f"{PROJECT_NAME}/"
    "model-package-v2/"
    "model.tar.gz"
)

# Optional local notebook download folder
LOCAL_DOWNLOAD_DIR = Path("downloaded_models")


# ============================================================
# AWS CLIENTS
# ============================================================

sm_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
s3_resource = boto3.resource("s3", region_name=REGION)


# ============================================================
# HELPER: Parse an S3 URI
# ============================================================

def parse_s3_uri(s3_uri: str):
    """
    Convert:
        s3://bucket-name/path/to/object
    into:
        bucket-name, path/to/object
    """
    parsed = urlparse(s3_uri)

    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")

    return parsed.netloc, parsed.path.lstrip("/")


# ============================================================
# STEP 1: Endpoint -> Endpoint Config -> SageMaker Model
# ============================================================

endpoint_desc = sm_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)

endpoint_config_name = endpoint_desc["EndpointConfigName"]

endpoint_config_desc = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config_name
)

production_variants = endpoint_config_desc["ProductionVariants"]

if not production_variants:
    raise ValueError("No production variants found in endpoint configuration.")

model_name = production_variants[0]["ModelName"]

model_desc = sm_client.describe_model(
    ModelName=model_name
)

print("Endpoint name:", ENDPOINT_NAME)
print("Endpoint status:", endpoint_desc["EndpointStatus"])
print("Endpoint configuration:", endpoint_config_name)
print("SageMaker model:", model_name)


# ============================================================
# STEP 2: SageMaker Model -> Model Package
# ============================================================

containers = model_desc.get("Containers", [])

if not containers:
    raise ValueError(
        "No Containers found in SageMaker model definition. "
        "Expected a model created from a Model Package."
    )

model_package_arn = containers[0].get("ModelPackageName")

if not model_package_arn:
    raise ValueError(
        "This model does not contain ModelPackageName. "
        "Inspect model_desc manually for a direct ModelDataUrl."
    )

print("Model package ARN:", model_package_arn)

package_desc = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

package_containers = package_desc["InferenceSpecification"]["Containers"]

if not package_containers:
    raise ValueError("No inference containers found in model package.")

model_s3_uri = package_containers[0].get("ModelDataUrl")

if not model_s3_uri:
    raise ValueError(
        "ModelDataUrl not found in model package inference container.\n"
        + json.dumps(package_containers[0], indent=2, default=str)
    )

print("\nOriginal model artefact S3 URI:")
print(model_s3_uri)


# ============================================================
# STEP 3: Verify source object exists
# ============================================================

source_bucket, source_key = parse_s3_uri(model_s3_uri)

source_metadata = s3_client.head_object(
    Bucket=source_bucket,
    Key=source_key
)

source_size_mb = source_metadata["ContentLength"] / (1024 * 1024)

print("\nSource bucket:", source_bucket)
print("Source key:", source_key)
print(f"Source model size: {source_size_mb:.2f} MB")


# ============================================================
# STEP 4: Download locally to the Studio notebook environment
# ============================================================

LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

local_model_file = LOCAL_DOWNLOAD_DIR / "model.tar.gz"

print("\nDownloading model locally...")
s3_client.download_file(
    source_bucket,
    source_key,
    str(local_model_file)
)

print("Local file saved to:")
print(local_model_file.resolve())

print(f"Local file size: {local_model_file.stat().st_size / (1024 * 1024):.2f} MB")


# ============================================================
# STEP 5: Copy the model directly into your team S3 prefix
# ============================================================

copy_source = {
    "Bucket": source_bucket,
    "Key": source_key
}

print("\nCopying model into team S3 folder...")

s3_resource.meta.client.copy(
    CopySource=copy_source,
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

destination_s3_uri = f"s3://{DESTINATION_BUCKET}/{DESTINATION_KEY}"

print("\nCopy completed successfully.")
print("Destination model artefact:")
print(destination_s3_uri)


# ============================================================
# STEP 6: Verify destination object
# ============================================================

destination_metadata = s3_client.head_object(
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

print("\nDestination verification:")
print("Destination size (MB):", round(
    destination_metadata["ContentLength"] / (1024 * 1024),
    2
))
print("Last modified:", destination_metadata["LastModified"])
print("ETag:", destination_metadata["ETag"])

Endpoint name: iti113-team14-airbnb-instant-booking
Endpoint status: InService
Endpoint configuration: iti113-team14-airbnb-instant-booking
SageMaker model: team14-airbnb-instant-booking-2026-08-21-17-05-47-483
Model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team14-airbnb-instant-booking/29

Original model artefact S3 URI:
s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-21-16-52-59-754/pipelines-059euzj1rjtu-RegisterModel-Repack-EqaR9hFOqW/output/model.tar.gz

Source bucket: sagemaker-ap-southeast-1-044528205969
Source key: sagemaker-scikit-learn-2026-08-21-16-52-59-754/pipelines-059euzj1rjtu-RegisterModel-Repack-EqaR9hFOqW/output/model.tar.gz
Source model size: 65.75 MB



Local file saved to:
/home/sagemaker-user/JG-Assignment/downloaded_models/model.tar.gz
Local file size: 65.75 MB

Copying model into team S3 folder...



Copy completed successfully.
Destination model artefact:
s3://nyp-26s1-iti113/iti113/team14/models/airbnb-instant-booking/model-package-v2/model.tar.gz

Destination verification:
Destination size (MB): 65.75
Last modified: 2026-08-21 17:08:53+00:00
ETag: "a5ec481da9cd227a32ffb9e761f21ccf-9"
